In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# 2. Gün ürettiğimiz RFM tablosunu yüklüyoruz
rfm = pd.read_csv('../data/processed/rfm_features.csv', index_col='Customer ID')
print(f"Toplam Müşteri Sayısı: {rfm.shape[0]}")
rfm.head()

Toplam Müşteri Sayısı: 4312


,recency,frequency,monetary,tenure,avg_order_value
Customer ID,,,,,
12346,166,11,372.86,362,33.90
12347,4,2,1323.32,41,661.66
12348,75,1,222.16,75,222.16
12349,44,3,2671.14,226,890.38
12351,12,1,300.93,12,300.93


In [2]:
# 1. Recency Skoru (Küçük gün = Yüksek puan 5)
rfm['recency_score'] = pd.qcut(rfm['recency'], 5, labels=[5, 4, 3, 2, 1])

# 2. Frequency Skoru (Yüksek sipariş = Yüksek puan 5)
rfm['frequency_score'] = pd.qcut(rfm['frequency'].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])

# 3. Monetary Skoru (Yüksek harcama = Yüksek puan 5)
rfm['monetary_score'] = pd.qcut(rfm['monetary'], 5, labels=[1, 2, 3, 4, 5])

# 4. Birleşik RF Skoru (Segment haritası için 2 haneli kod)
rfm['RF_SCORE'] = (rfm['recency_score'].astype(str) + rfm['frequency_score'].astype(str))

rfm.head()

,recency,frequency,monetary,tenure,avg_order_value,recency_score,frequency_score,monetary_score,RF_SCORE
Customer ID,,,,,,,,,
12346,166,11,372.86,362,33.90,2,5,2,25
12347,4,2,1323.32,41,661.66,5,2,4,52
12348,75,1,222.16,75,222.16,2,1,1,21
12349,44,3,2671.14,226,890.38,3,3,5,33
12351,12,1,300.93,12,300.93,5,1,2,51


In [3]:
seg_map = {
    r'[1-2][1-2]': 'hibernating',
    r'[1-2][3-4]': 'at_Risk',
    r'[1-2]5': 'cant_loose',
    r'3[1-2]': 'about_to_sleep',
    r'33': 'need_attention',
    r'[3-4][4-5]': 'loyal_customers',
    r'41': 'promising',
    r'51': 'new_customers',
    r'[4-5][2-3]': 'potential_loyalists',
    r'5[4-5]': 'champions'
}

rfm['segment'] = rfm['RF_SCORE'].replace(seg_map, regex=True)

rfm.head()

,recency,frequency,monetary,tenure,avg_order_value,recency_score,frequency_score,monetary_score,RF_SCORE,segment
Customer ID,,,,,,,,,,
12346,166,11,372.86,362,33.90,2,5,2,25,cant_loose
12347,4,2,1323.32,41,661.66,5,2,4,52,potential_loyalists
12348,75,1,222.16,75,222.16,2,1,1,21,hibernating
12349,44,3,2671.14,226,890.38,3,3,5,33,need_attention
12351,12,1,300.93,12,300.93,5,1,2,51,new_customers


In [4]:
segment_summary = rfm.groupby('segment').agg({
    'recency': ['mean', 'count'],
    'frequency': ['mean'],
    'monetary': ['mean', 'sum'],
    'avg_order_value': ['mean']
})

segment_summary.columns = ['recency_mean', 'customer_count', 'frequency_mean', 'monetary_mean', 'monetary_total', 'avg_order_value_mean']
segment_summary['customer_share_%'] = (segment_summary['customer_count'] / len(rfm)) * 100
segment_summary['revenue_share_%'] = (segment_summary['monetary_total'] / rfm['monetary'].sum()) * 100

segment_summary.sort_values(by='customer_count', ascending=False)

,recency_mean,customer_count,frequency_mean,monetary_mean,monetary_total,avg_order_value_mean,customer_share_%,revenue_share_%
segment,,,,,,,,
hibernating,214.88,1015,1.13,403.98,410037.50,354.45,23.54,4.64
loyal_customers,37.29,742,6.83,2746.07,2037581.98,387.47,17.21,23.07
champions,8.12,663,12.55,6852.26,4543051.14,413.25,15.38,51.44
at_Risk,153.16,611,3.07,1188.88,726404.65,370.32,14.17,8.22
potential_loyalists,19.79,517,2.02,729.51,377157.18,353.56,11.99,4.27
about_to_sleep,54.82,343,1.20,441.32,151372.76,367.96,7.95,1.71
need_attention,54.27,207,2.45,1060.36,219493.90,424.29,4.80,2.49
promising,26.75,87,1.00,367.09,31936.55,367.09,2.02,0.36
cant_loose,125.12,77,9.12,4099.45,315657.65,464.04,1.79,3.57


In [5]:
import os

output_path = os.path.abspath("../data/processed/rfm_segmented.csv")
rfm.to_csv(output_path)
print(f"3. Günün segmentlenmiş müşteri tablosu başarıyla kaydedildi: {output_path}")

3. Günün segmentlenmiş müşteri tablosu başarıyla kaydedildi: C:\Users\USER\ecommerce-intelligence\data\processed\rfm_segmented.csv
